# Model Exploration

Quick exploratory checks before training the full Spark pipeline:

- Target distribution (next-day returns)
- Feature correlations
- A trivial baseline: how well does "always predict up" do?

Run `python -m src.data_ingest` and `python -m src.features` first so `data/features/features.parquet` exists.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_parquet("../data/features/features.parquet")
print(df.shape)
df.head()

## Target distribution

Daily returns should be roughly centered at zero with fat tails. If the mean is far from zero, something is wrong with the data.

In [ ]:
df["next_return"].hist(bins=100)
plt.xlabel("Next-day return")
plt.ylabel("Count")
plt.title("Distribution of next-day returns")
plt.show()

print(df["next_return"].describe())

## Baseline: always predict "up"

If a model cannot beat this trivial baseline, it has learned nothing useful.

In [ ]:
baseline_acc = (df["label_up"] == 1).mean()
print(f"Baseline (always predict up) accuracy: {baseline_acc:.3f}")

# And a mean-return baseline for the regression task
mean_return = df["next_return"].mean()
baseline_mae = (df["next_return"] - mean_return).abs().mean()
print(f"Baseline (predict mean return) MAE: {baseline_mae:.5f}")

## Feature correlations

Check which features actually move with the target. High correlation between features themselves (e.g. the moving averages) is expected.

In [ ]:
feature_cols = [c for c in df.columns if c.startswith(("ma_ratio", "vol_ma", "return_lag"))]
feature_cols += ["daily_return", "volatility", "rsi", "market_prob_up", "market_edge", "market_confidence"]

corr = df[feature_cols + ["next_return"]].corr()["next_return"].drop("next_return")
corr.sort_values().plot(kind="barh")
plt.title("Correlation of each feature with next-day return")
plt.tight_layout()
plt.show()

## Prediction-market signal sanity check

The mock market probability should be weakly positively related to the actual next-day direction. If this correlation is zero or negative, the mock generator needs a look.

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(df["label_up"], df["market_prob_up"])
print(f"Mock market probability AUC vs actual direction: {auc:.3f}")
print("(Anything clearly above 0.50 means the mock carries signal, as designed.)")